In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ChicagoTaxi-ETL") \
    .getOrCreate()


26/01/02 17:43:35 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# Load Raw Data from GS

In [2]:
INPUT_PATH = "gs://joan-ngugi/chicago_taxi/raw"

df = spark.read.parquet(INPUT_PATH)
print("Rows:", df.count())
df.printSchema()


Rows: 10583626
root
 |-- unique_key: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp_ntz (nullable = true)
 |-- trip_end_timestamp: timestamp_ntz (nullable = true)
 |-- trip_seconds: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: long (nullable = true)
 |-- dropoff_census_tract: long (nullable = true)
 |-- pickup_community_area: long (nullable = true)
 |-- dropoff_community_area: long (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nul

# Data Cleaning & Feature Engineering

In [4]:
from pyspark.sql.functions import col


df_clean = df.filter(
    (col("trip_seconds") > 60) &
    (col("trip_miles") > 0.1) &
    (col("fare") > 0)
)


In [5]:
from pyspark.sql.functions import round

df_features = df_clean \
    .withColumn("trip_minutes", round(col("trip_seconds") / 60, 2)) \
    .withColumn("fare_per_mile", round(col("fare") / col("trip_miles"), 2))


In [6]:
df_features.limit(20).toPandas()


,unique_key,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,payment_type,company,pickup_latitude,pickup_longitude,pickup_location,dropoff_latitude,dropoff_longitude,dropoff_location,trip_minutes,fare_per_mile
0,958ff10f886862f1c029e46a7278a74683de1d58,7f83e923d71df8812353e1ea8f491ca1cc70469fe9ed27...,2017-02-19 12:30:00,2017-02-19 12:45:00,780,5.30,NaN,NaN,NaN,NaN,...,Cash,City Service,NaN,NaN,None,NaN,NaN,None,13.00,3.02
1,efa0ad76418eab134be1f30c137a6b3750e3a016,e3a512583c77aab0f5097330d49e1401e4e1a581b0b5c5...,2015-12-10 08:15:00,2015-12-10 08:15:00,540,1.20,NaN,NaN,NaN,NaN,...,Credit Card,None,NaN,NaN,None,NaN,NaN,None,9.00,5.88
2,2aead0670bc9f7c2f1536ec68925525c2475ec10,7f83e923d71df8812353e1ea8f491ca1cc70469fe9ed27...,2013-12-07 13:00:00,2013-12-07 13:00:00,480,1.10,NaN,NaN,NaN,NaN,...,Cash,None,NaN,NaN,None,NaN,NaN,None,8.00,6.05
3,e161e1c3a275975159de4ceb9b0b81cef4a48e33,c0a216e4b6cea5aefd65a6a940125df31dde6b1bdd48e5...,2013-12-22 13:00:00,2013-12-22 13:00:00,480,1.70,NaN,NaN,NaN,NaN,...,Cash,None,NaN,NaN,None,NaN,NaN,None,8.00,4.38
4,de9c3f74730a8e47f38bac2d5e5c50c4913499d8,e0b3e1e1ca6320c46aa1688ef894aeae326f89ae0de0f4...,2021-02-17 17:15:00,2021-02-17 17:30:00,600,4.50,NaN,NaN,NaN,NaN,...,Credit Card,Top Cab Affiliation,NaN,NaN,None,NaN,NaN,None,10.00,3.06
5,faec73a25db2221820b483d8ff4bd27894e90845,636f6cfd7a366daa3ae1bb4ca25eb4b08a670f838d4cd9...,2016-02-19 08:30:00,2016-02-19 08:30:00,446,2.00,NaN,NaN,NaN,NaN,...,Cash,Norshore Cab,NaN,NaN,None,NaN,NaN,None,7.43,3.70
6,a77db77fa8d32da20f5261f3a9f80b668b49a4db,136bde683344e8d6f81bb0a493c99e3176256fd2564687...,2019-06-04 20:00:00,2019-06-04 20:15:00,420,0.72,NaN,NaN,NaN,NaN,...,Credit Card,Patriot Taxi Dba Peace Taxi Associat,NaN,NaN,None,NaN,NaN,None,7.00,6.94
7,8b1948e238c2fea52f9e46e47e01cd3bce3288cd,fd2d4d150336dbc81f9b26fd2e2721b8652540d2692981...,2014-01-21 21:15:00,2014-01-21 21:30:00,660,2.60,NaN,NaN,NaN,NaN,...,Cash,None,NaN,NaN,None,NaN,NaN,None,11.00,3.48
8,d5cbd1371da61fe9a0d2fee9ce3394cbc8167259,f6db72e04ae8ef4d6fd82c21bacd9599a3487e7ba5b829...,2014-02-03 07:45:00,2014-02-03 07:45:00,780,3.90,NaN,NaN,NaN,NaN,...,Cash,None,NaN,NaN,None,NaN,NaN,None,13.00,2.88
9,7bb2054ef39f84ddb17eef5a073ff1fd2382e198,5b0e845181646f6a7a9201c1edc9a7b0d4aa0fd14bf255...,2016-08-27 21:15:00,2016-08-27 21:45:00,1380,4.30,NaN,NaN,NaN,NaN,...,Cash,Sun Taxi,NaN,NaN,None,NaN,NaN,None,23.00,3.72


In [7]:
#Save Processed Data
OUTPUT_PATH = "gs://joan-ngugi/chicago_taxi/processed/"

df_features.write.mode("overwrite").parquet(OUTPUT_PATH)


# Machine Learning with PySpark ML

In [8]:
# Load Processed Data
df = spark.read.parquet("gs://joan-ngugi/chicago_taxi/processed/")


In [9]:
from pyspark.sql.functions import when

df = df.withColumn(
    "is_high_fare",
    when(col("fare") >= 20, 1).otherwise(0)
)


In [10]:
features = [
    "trip_miles",
    "trip_minutes",
    "fare_per_mile"
]

df_ml = df.select(features + ["is_high_fare"])


In [11]:
df_ml.show()

+----------+------------+-------------+------------+
|trip_miles|trip_minutes|fare_per_mile|is_high_fare|
+----------+------------+-------------+------------+
|      16.4|       10.88|         5.02|           1|
|      0.89|        15.0|         9.83|           0|
|     14.96|        41.0|         2.61|           1|
|       5.2|        14.0|         2.66|           0|
|      5.64|        34.0|         3.27|           0|
|       1.0|        11.0|         7.25|           0|
|       0.6|         3.0|         6.08|           0|
|      10.8|        47.6|          0.0|           0|
|       2.0|       13.75|          5.0|           0|
|      14.7|       30.98|          2.6|           1|
|       1.4|         6.0|         4.46|           0|
|       4.3|        13.0|          2.9|           0|
|      1.53|        26.0|        20.42|           1|
|       1.3|         5.0|          4.5|           0|
|       3.6|        18.0|         3.29|           0|
|      18.5|        23.0|         2.42|       

In [12]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)

df_ml = assembler.transform(df_ml) \
    .select("features", col("is_high_fare").alias("label"))


In [13]:
train, test = df_ml.randomSplit([0.7, 0.3], seed=42)


In [14]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    numTrees=20,
    maxDepth=5,
    seed=42
)

model = rf.fit(train)


In [15]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

predictions = model.transform(test)

evaluator = BinaryClassificationEvaluator()
auc = evaluator.evaluate(predictions)

print("AUC:", auc)


AUC: 0.9972108127222252


In [16]:
# Save Predictions
PRED_PATH = "gs://joan-ngugi/chicago_taxi/predictions/"

predictions.select(
    "label", "prediction", "probability"
).write.mode("overwrite").parquet(PRED_PATH)
